# Modeling

This notebook builds and evaluates Remaining Useful Life (RUL) regression models for the NASA C-MAPSS FD001 dataset.

The workflow is intentionally simple:

1. Load the train, test, and test RUL files.
2. Build the training RUL target.
3. Define the feature set.
4. Split training engines into train and validation groups.
5. Compare simple models using linear RUL.
6. Repeat the comparison with capped RUL.
7. Select one approach.
8. Evaluate it once on the NASA test set.
9. Inspect prediction quality and the largest errors.


## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split


In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "CMAPSSData"

TRAIN_PATH = DATA_DIR / "train_FD001.txt"
TEST_PATH = DATA_DIR / "test_FD001.txt"
TEST_RUL_PATH = DATA_DIR / "RUL_FD001.txt"


In [ ]:
columns = (
    ["unit_id", "cycle"]
    + [f"setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)


In [ ]:
train_df = pd.read_csv(
    TRAIN_PATH,
    sep=r"\s+",
    header=None,
    names=columns,
)

test_df = pd.read_csv(
    TEST_PATH,
    sep=r"\s+",
    header=None,
    names=columns,
)

test_rul = pd.read_csv(
    TEST_RUL_PATH,
    sep=r"\s+",
    header=None,
    names=["rul"],
)


## 2. Test set structure

Training trajectories run until failure, while test trajectories stop before failure.

`RUL_FD001.txt` contains one true RUL value for each test engine at its last observed cycle. The table below connects each test engine with that final observed cycle and its provided RUL.


In [ ]:
test_last_cycles = (
    test_df.groupby("unit_id")["cycle"]
    .max()
    .reset_index(name="last_observed_cycle")
)

test_summary = test_last_cycles.copy()
test_summary["true_rul"] = test_rul["rul"].to_numpy()

test_summary.head()


## 3. Training target

Because every training trajectory reaches failure, the RUL of a training observation can be calculated as:

`RUL = final cycle of the engine - current cycle`

The final observation of each training engine therefore has RUL equal to zero.


In [ ]:
train_max_cycles = train_df.groupby("unit_id")["cycle"].transform("max")

train_df["rul"] = train_max_cycles - train_df["cycle"]

train_df.loc[
    train_df["unit_id"] == 1,
    ["unit_id", "cycle", "rul"],
].tail()


## 4. Features

The EDA identified several constant columns. These columns are removed because they do not provide information that can help distinguish observations.

`unit_id` is also excluded because it is an identifier, not a predictive feature.

`cycle` is kept as a feature because engine age can carry useful information for RUL estimation.


In [ ]:
constant_columns = [
    "setting_3",
    "sensor_1",
    "sensor_5",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19",
]

feature_columns = [
    column
    for column in train_df.columns
    if column not in constant_columns + ["unit_id", "rul"]
]


## 5. Train-validation split

The split is performed by engine instead of by individual rows.

This prevents observations from the same engine from appearing in both training and validation data.


In [ ]:
engine_ids = train_df["unit_id"].unique()

train_engine_ids, val_engine_ids = train_test_split(
    engine_ids,
    test_size=0.2,
    random_state=42,
)

train_data = train_df[
    train_df["unit_id"].isin(train_engine_ids)
].copy()

val_data = train_df[
    train_df["unit_id"].isin(val_engine_ids)
].copy()


In [ ]:
X_train = train_data[feature_columns]
y_train = train_data["rul"]

X_val = val_data[feature_columns]
y_val = val_data["rul"]


## 6. Linear RUL experiments

We first keep the original linear RUL target and compare three models:

- `DummyRegressor`: reference baseline.
- `LinearRegression`: simple linear model.
- `RandomForestRegressor`: non-linear tree ensemble.

MAE is interpreted directly in cycles. RMSE penalizes large prediction errors more strongly.


In [ ]:
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_val)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_val)

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_val)


In [ ]:
dummy_mae = mean_absolute_error(y_val, dummy_predictions)
dummy_rmse = mean_squared_error(y_val, dummy_predictions) ** 0.5

linear_mae = mean_absolute_error(y_val, linear_predictions)
linear_rmse = mean_squared_error(y_val, linear_predictions) ** 0.5

rf_mae = mean_absolute_error(y_val, rf_predictions)
rf_rmse = mean_squared_error(y_val, rf_predictions) ** 0.5

linear_results = pd.DataFrame(
    {
        "model": [
            "DummyRegressor",
            "LinearRegression",
            "RandomForest",
        ],
        "mae": [
            dummy_mae,
            linear_mae,
            rf_mae,
        ],
        "rmse": [
            dummy_rmse,
            linear_rmse,
            rf_rmse,
        ],
    }
)

linear_results.round(2)


### Linear RUL model comparison

This chart answers one question: **which model performs better when all models use the same linear RUL target?**

Lower bars are better.


In [ ]:
linear_plot = linear_results.set_index("model")[["mae", "rmse"]]

linear_plot.plot(
    kind="bar",
    figsize=(9, 5),
    rot=0,
)

plt.ylabel("Error (cycles)")
plt.xlabel("")
plt.title("Validation metrics — Linear RUL")
plt.legend(["MAE", "RMSE"])
plt.tight_layout()
plt.show()


The learned models clearly improve on the dummy baseline. Random Forest slightly reduces MAE compared with Linear Regression, while their RMSE values remain similar.


## 7. Capped RUL experiments

The linear target assumes that exact RUL should be estimated from the beginning of an engine's life.

We now test a capped target:

`capped RUL = min(linear RUL, 125)`

The cap represents an initial healthy region in which distinguishing very large RUL values is less important. The same features, engine split, and models are kept so that the experiment changes only the target formulation.


In [ ]:
RUL_CAP = 125

y_train_capped = train_data["rul"].clip(upper=RUL_CAP)
y_val_capped = val_data["rul"].clip(upper=RUL_CAP)


In [ ]:
dummy_capped = DummyRegressor(strategy="mean")
dummy_capped.fit(X_train, y_train_capped)
dummy_capped_predictions = dummy_capped.predict(X_val)

linear_capped = LinearRegression()
linear_capped.fit(X_train, y_train_capped)
linear_capped_predictions = linear_capped.predict(X_val)

rf_capped = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)
rf_capped.fit(X_train, y_train_capped)
rf_capped_predictions = rf_capped.predict(X_val)


In [ ]:
dummy_capped_mae = mean_absolute_error(
    y_val_capped,
    dummy_capped_predictions,
)
dummy_capped_rmse = mean_squared_error(
    y_val_capped,
    dummy_capped_predictions,
) ** 0.5

linear_capped_mae = mean_absolute_error(
    y_val_capped,
    linear_capped_predictions,
)
linear_capped_rmse = mean_squared_error(
    y_val_capped,
    linear_capped_predictions,
) ** 0.5

rf_capped_mae = mean_absolute_error(
    y_val_capped,
    rf_capped_predictions,
)
rf_capped_rmse = mean_squared_error(
    y_val_capped,
    rf_capped_predictions,
) ** 0.5

capped_results = pd.DataFrame(
    {
        "model": [
            "DummyRegressor",
            "LinearRegression",
            "RandomForest",
        ],
        "mae": [
            dummy_capped_mae,
            linear_capped_mae,
            rf_capped_mae,
        ],
        "rmse": [
            dummy_capped_rmse,
            linear_capped_rmse,
            rf_capped_rmse,
        ],
    }
)

capped_results.round(2)


### Capped RUL model comparison

This chart answers the same question as before, now **within the capped RUL formulation**.

The metric values from the linear and capped targets should not be compared as if they represented exactly the same regression problem, because the target distribution changed.


In [ ]:
capped_plot = capped_results.set_index("model")[["mae", "rmse"]]

capped_plot.plot(
    kind="bar",
    figsize=(9, 5),
    rot=0,
)

plt.ylabel("Error (cycles)")
plt.xlabel("")
plt.title("Validation metrics — Capped RUL")
plt.legend(["MAE", "RMSE"])
plt.tight_layout()
plt.show()


Within the capped RUL experiment, Random Forest achieves the lowest MAE and RMSE among the tested models.

For the remainder of this notebook, we use:

- **Target:** capped RUL at 125 cycles
- **Model:** Random Forest
- **Validation strategy:** split by engine

This is the selected experimental configuration, not a claim that more complex models or additional tuning could not improve the result.


## 8. Final evaluation on the NASA test set

The test set is used only after selecting the experimental configuration.

Each test engine is truncated before failure, so evaluation is performed at its **last observed cycle**. The true RUL for that point comes from `RUL_FD001.txt`.


In [ ]:
test_last_rows = (
    test_df.sort_values(["unit_id", "cycle"])
    .groupby("unit_id")
    .tail(1)
    .sort_values("unit_id")
)

X_test = test_last_rows[feature_columns]
y_test = test_rul["rul"].to_numpy()

test_predictions = rf_capped.predict(X_test)

test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = mean_squared_error(y_test, test_predictions) ** 0.5

print(f"Test MAE: {test_mae:.2f} cycles")
print(f"Test RMSE: {test_rmse:.2f} cycles")


In [ ]:
test_results = pd.DataFrame(
    {
        "unit_id": test_last_rows["unit_id"].to_numpy(),
        "true_rul": y_test,
        "predicted_rul": test_predictions,
    }
)

test_results["error"] = (
    test_results["predicted_rul"]
    - test_results["true_rul"]
)

test_results["absolute_error"] = test_results["error"].abs()


### Prediction quality

The chart below compares the model prediction with the true RUL for each of the 100 test engines.

- Points on the diagonal represent perfect predictions.
- Points above the diagonal mean that the model **overestimated** remaining life.
- Points below the diagonal mean that the model **underestimated** remaining life.

This is the main visual summary of final test performance.


In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    test_results["true_rul"],
    test_results["predicted_rul"],
    alpha=0.7,
)

min_value = min(
    test_results["true_rul"].min(),
    test_results["predicted_rul"].min(),
)

max_value = max(
    test_results["true_rul"].max(),
    test_results["predicted_rul"].max(),
)

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--",
)

plt.xlabel("True RUL (cycles)")
plt.ylabel("Predicted RUL (cycles)")
plt.title("NASA test set — True vs Predicted RUL")
plt.tight_layout()
plt.show()


## 9. Error analysis

Rather than adding several disconnected plots, the error analysis focuses on one concrete question:

**Which test engines produced the largest mistakes, and did the model overestimate or underestimate their remaining life?**

`error = predicted RUL - true RUL`

- Positive error: remaining life was overestimated.
- Negative error: remaining life was underestimated.


In [ ]:
largest_errors = (
    test_results
    .sort_values("absolute_error", ascending=False)
    .head(10)
)

largest_errors[
    [
        "unit_id",
        "true_rul",
        "predicted_rul",
        "error",
        "absolute_error",
    ]
].round(2)


## 10. Conclusion

The experiment established a complete baseline workflow for FD001:

- RUL was constructed from the run-to-failure training trajectories.
- Constant features and the engine identifier were excluded.
- Train and validation data were separated by engine to avoid row-level leakage.
- Dummy, Linear Regression, and Random Forest models were compared.
- Linear and capped RUL formulations were tested separately.
- Random Forest with capped RUL was selected for final evaluation.
- Final performance was measured on the NASA test set at the last observed cycle of each engine.
- Prediction quality was inspected using one true-vs-predicted chart and the largest individual errors.

The next project phase is no longer exploratory modeling. The next step is to move the selected workflow out of the notebook and into reusable training and evaluation code.
